In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

# ※ Quiz : 경주여행과 전주여행에 대해 최빈단어 시각화 유사도 분석
- (1) naver open API를 활용하여 블로그에 "경주여행", "전주여행"을 각각 500건씩 검색하여 백업(data/quiz/naver.csv)
    * 파일 내용 : query, no, title, link, description, total_text(title + ' ' + description)
- (2) naver.csv에서 total_text를 품사태깅(naver_pos.csv)
    * 파일 내용 : query, no, token, pos
- (3) 명사만 추출(naver_pos_nouns.csv)
    * query, token, pos
- (4) 빈도분석 백업(naver_pos_nouns_count.csv)
    * token, 경주빈도, 전주빈도, 빈도합
- (5) 빈도 시각화(워드클라우드, Text.plot)
    * 이미지 저장
- (6) 단어간 거리 분석(Word2Vec)

# 1. 네이버 open API 활용하여 검색 추출
- query, no, title, link, description, total_text(title + ' ' + description)

In [1]:
# .env가져오기
from dotenv import load_dotenv
import os
load_dotenv()

True

In [2]:
# 네이버 개발자 센터에 있는 소스를 가져오기
# 네이버 검색 API 예제 - 블로그 검색
import os
import sys
import urllib.request
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')
encText = urllib.parse.quote("경주 여행")
url = "https://openapi.naver.com/v1/search/blog.json?query=" + encText # JSON 결과
# url = "https://openapi.naver.com/v1/search/blog.xml?query=" + encText # XML 결과
request = urllib.request.Request(url)
request.add_header("X-Naver-Client-Id",client_id)
request.add_header("X-Naver-Client-Secret",client_secret)
response = urllib.request.urlopen(request)
rescode = response.getcode()
if(rescode==200):
    response_body = response.read()
    print(response_body.decode('utf-8')[:200])
    # items = json.loads(response_body.decode('utf-8'))
else:
    print("Error Code:" + rescode)

{
	"lastBuildDate":"Fri, 04 Sep 2026 12:49:24 +0900",
	"total":2737577,
	"start":1,
	"display":10,
	"items":[
		{
			"title":"3월의 <b>경주여행<\/b>.",
			"link":"https:\/\/lje77777.tistory.com\/7132",
			"


In [3]:
# 문자->dict
import json
from html import unescape # description에 있는 &lt;(특수문자)를 <로 변경
import requests
import pandas as pd
import re  # 특수문자 제거 @@ _._

In [4]:
query = "경주 여행"
start = 1
#url = f"https://openapi.naver.com/v1/search/blog.json?query={query}&display=100&start={start}"
url = "https://openapi.naver.com/v1/search/blog.json"
params = {
    'query':query,
    'display':100,
    'start':start
}
headers = {
    "X-Naver-Client-Id":client_id,
    "X-Naver-Client-Secret":client_secret
}
response = requests.get(url, headers=headers, params=params)
# 문자 -> dict
# items = json.loads(response.text)['items']
items = response.json()['items']
items[4]

{'title': '경북 <b>경주여행</b>/불국사-단아함과 정숙함이 묻어나는 겹벚꽃(왕벚꽃)....',
 'link': 'https://skdywjd25.tistory.com/5973',
 'description': '벚꽃이 다 지고나면 피는 겹벗꽃 봄 <b>여행</b>을 즐기는 고수라면 서둘러 <b>경주</b>행 티켓을 예약하자. 또다시 <b>경주</b>가 들썩이고 있다.다 <b>경주</b>지역 벚꽃은 거의 떨어지고 불국사 쪽이 겹벚꽃으로 유명하다. 불국사 입구에... ',
 'bloggername': '산행과여행 사진으로 말한다',
 'bloggerlink': 'https://skdywjd25.tistory.com/',
 'postdate': '20190425'}

In [5]:
# title과 description의 <b>없애기, html의 특수문자없애기, 일반특수 없애기
item = items[4]
title = item['title'].replace('<b>', ' ').replace('</b>', ' ')
title = unescape(title)
title = re.sub(r'[^a-zA-Z0-9가-힣]', ' ', title)
description = item['description'].replace('<b>', ' ').replace('</b>', ' ')
description = unescape(description)
description = re.sub(r'[^a-zA-Z0-9가-힣]', ' ', description)
link = item['link']
totaltext = title + ' ' + description
print(totaltext)
print(link)

경북  경주여행  불국사 단아함과 정숙함이 묻어나는 겹벚꽃 왕벚꽃      벚꽃이 다 지고나면 피는 겹벗꽃 봄  여행 을 즐기는 고수라면 서둘러  경주 행 티켓을 예약하자  또다시  경주 가 들썩이고 있다 다  경주 지역 벚꽃은 거의 떨어지고 불국사 쪽이 겹벚꽃으로 유명하다  불국사 입구에    
https://skdywjd25.tistory.com/5973


In [6]:
# re 정규표현식을 이용해서 특수문자 없애기
title = '[여행] ## & ktx 타고 짱 ㅋㅋ ㅠㅠ'
re.sub(r'[^a-zA-Z0-9가-힣]', ' ', title)

' 여행       ktx 타고 짱      '

In [10]:
# 네이버 API 정보 및 검색 정보
from dotenv import load_dotenv
import os
load_dotenv()
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')

In [12]:
def get_search_item_return(query, start):
    'query, no, title, link, description, total_text(title + ' ' + description)'
    import requests
    # import json
    from html import unescape # html 특수문자 제거(ex. &nbsp; 제거)
    import re # 일반 특수문제 제거(__.__제거 )
    from dotenv import load_dotenv
    import os
    load_dotenv() # .env파일 읽어오기(환경변수 설정)
    client_id = os.getenv('CLIENT_ID')
    client_secret = os.getenv('CLIENT_SECRET')
    url = "https://openapi.naver.com/v1/search/blog.json"
    params = {
        'query':query,
        'display':100,
        'start':start
    }
    headers = {
        "X-Naver-Client-Id":client_id,
        "X-Naver-Client-Secret":client_secret
    }
    response = requests.get(url, headers=headers, params=params)
    # 문자 -> dict
    # items = json.loads(response.text)['items']
    items = response.json()['items'] # 100개
    result = [] # items의 title,link,description등을 dict list로 append
    for i, item in enumerate(items):        
        no = (start-1)*100 + i+1 # start가 1일때는 1,2,3.. start가 2일때는 101, 102,..
        title = item['title'].replace('<b>', ' ').replace('</b>', ' ')
        title = unescape(title)
        title = re.sub(r'[^a-zA-Z0-9가-힣]', ' ', title)
        link = item['link']
        description = item['description'].replace('<b>', ' ').replace('</b>', ' ')
        description = unescape(description)
        description = re.sub(r'[^a-zA-Z0-9가-힣]', ' ', description)
        total_text = title + ' ' + description
        #print(query, no, title[:5], link, description[:5], total_text[:5])
        result.append({'query': query,
                      'no'    : no,
                      'title' : title,
                      'link'  : link,
                      'description':description,
                      'total_text' :total_text
                      })
    return result

In [13]:
import time
queries = ['경주 여행', '전주 여행']
max_start = 5
result_total = []
for query in queries:
    for start in range(1, max_start+1):
        print(f'{query} 읽어오는 중...{start}/{max_start}')
        result_total.extend(get_search_item_return(query, start))
        time.sleep(0.5)

경주 여행 읽어오는 중...1/5
경주 여행 읽어오는 중...2/5
경주 여행 읽어오는 중...3/5
경주 여행 읽어오는 중...4/5
경주 여행 읽어오는 중...5/5
전주 여행 읽어오는 중...1/5
전주 여행 읽어오는 중...2/5
전주 여행 읽어오는 중...3/5
전주 여행 읽어오는 중...4/5
전주 여행 읽어오는 중...5/5


In [14]:
import pandas as pd
df = pd.DataFrame(result_total)
print(df.shape)
df.loc[498:501]

(1000, 6)


,query,no,title,link,description,total_text
498,경주 여행,499,가을 경주 여행,https://lje77777.tistory.com/7492,경주 도착 오늘은 작년 여름 경주여행 코스였던 서출지와 통일전 그리고 산림 ...,가을 경주 여행 경주 도착 오늘은 작년 여름 경주여행 코스였던 서출지와...
499,경주 여행,500,알라딘서재 100자평 일상이 고고학 나 혼자 경주 여행,https://blog.aladin.co.kr/787898106/12069804,일상이 고고학 나 혼자 경주 여행 개정증보판 일상이 고고학 시리즈 2 ...,알라딘서재 100자평 일상이 고고학 나 혼자 경주 여행 일상이 고고학 ...
500,전주 여행,1,알라딘서재 전주 당일 여행,https://blog.aladin.co.kr/melldy/5785245,전주여행 20120707 버스를 타고 마음이 두근두근두근 음악도 들으면서 핸...,알라딘서재 전주 당일 여행 전주여행 20120707 버스를 타고 마음이 ...
501,전주 여행,2,전주여행 전주 한옥마을 아카 갤러리카페 Jeonju Hanok Vill...,https://desert.tistory.com/6371,전주여행 전주 한옥마을 아카 갤러리카페 전주 한옥마을 Jeonju H...,전주여행 전주 한옥마을 아카 갤러리카페 Jeonju Hanok Vill...


In [15]:
df.to_csv('data/quiz_naver.csv', encoding='cp949', index=False)

# 2. 품사 태깅 백업
- query, no, token, pos -> quiz_naver_pos.csv백업

In [16]:
df = pd.read_csv('data/quiz_naver.csv', encoding='cp949')
df.head(1)

,query,no,title,link,description,total_text
0,경주 여행,1,경주여행 동궁과월지 첨성대 야경 연꽃단지 분황사,https://green54.tistory.com/1244,경주여행 때 방문했던 곳들을 다시 한번 남겨보려함 경주여행 경주 밤...,경주여행 동궁과월지 첨성대 야경 연꽃단지 분황사 경주여행 때 방문했던 ...


In [17]:
df_list = df[['query','no','total_text']].values.tolist()
df_list[::500], len(df_list)

([['경주 여행',
   1,
   ' 경주여행   동궁과월지 첨성대 야경  연꽃단지  분황사  경주여행  때 방문했던 곳들을 다시 한번 남겨보려함     경주여행    경주 밤에가볼만한곳    이웃님들 잘 지내셨나요  지난주 일    blog naver com 예전  여행 사진을 보니  여행 을 떠나고싶어진다 '],
  ['전주 여행',
   1,
   ' 알라딘서재  전주  당일 여행   전주여행  20120707 버스를 타고 마음이 두근두근두근  음악도 들으면서  핸드폰도 바꾸고 버스에서 내리니 날씨가 너무 화창했어욤    버스가 안와서 한참을 기다린  버스타고 전동성당으로 향했어요  말로만듣던    ']],
 1000)

In [18]:
stopwords = ['전주', '여행', '경주']
'전주' not in stopwords

False

In [19]:
from konlpy.tag import Hannanum, Kkma, Komoran, Okt
from mecab import MeCab
analyzer = MeCab()
postagged_list = [] # 품사태깅한 dict list (query, no, token, pos)
stopwords = ['전주', '여행', '경주']
for i, row in enumerate(df_list):
    if i%250==0:
        print(f'총 1000번중 현재 {i}번째 품사태깅중')
    query = row[0]
    no    = row[1]
    total_text = row[2]
    tagged_list = analyzer.pos(total_text)
    for token, tag in tagged_list:
        if token not in stopwords and len(token)>1 : #2글자 이상의 stopword가 아닌 token
            postagged_list.append({'query':query,
                                  'no':no,
                                  'token':token,
                                  'pos':tag})

총 1000번중 현재 0번째 품사태깅중
총 1000번중 현재 250번째 품사태깅중
총 1000번중 현재 500번째 품사태깅중
총 1000번중 현재 750번째 품사태깅중


In [20]:
df_postagged = pd.DataFrame(postagged_list)
df_postagged.to_csv('data/quiz_naver_pos.csv', encoding='cp949', index=False)

# 3. 명사만 추출
- MeCab에서의 명사(NNG, NNP, NP)
- query, token, pos

In [21]:
df_postagged = pd.read_csv('data/quiz_naver_pos.csv', encoding='cp949')
select_pos = ['NNG', 'NNP', 'NP']
df_postagged.sample()

,query,no,token,pos
1852,경주 여행,74,마무리,NNG


In [22]:
df_nouns = df_postagged.loc[df_postagged['pos'].isin(select_pos), ['query','token','pos']]
df_nouns.sample()

,query,token,pos
23780,전주 여행,이번,NNG


In [23]:
df_nouns.to_csv('data/quiz_naver_pos_noun.csv', encoding='cp949', index=False)

# 4. 빈도분석
- token, 경주빈도, 전주빈도, 빈도합, 경주비율, 전주비율

In [24]:
df_nouns.groupby(['query'], as_index=False)['pos'].count()

,query,pos
0,경주 여행,8025
1,전주 여행,8978


In [25]:
df_token_grp = df_nouns.groupby(['query', 'token'], as_index=False).count()
#df_token_grp.columns = ['query', 'token', 'count']
df_token_grp.rename(columns={'pos':'count'}, inplace=True)
df_token_grp.sample()

,query,token,count
1114,전주 여행,시외버스,21


In [26]:
df_gj = df_token_grp.loc[df_token_grp['query']=='경주 여행', ['token','count']]
df_jj = df_token_grp.loc[df_token_grp['query']=='전주 여행', ['token','count']]
df_gj.shape, df_jj.shape

((708, 2), (812, 2))

In [27]:
df_gj.head(2)

,token,count
0,가게,2
1,가격,11


In [28]:
df_jj.head(2)

,token,count
708,가게,18
709,가격,11


In [29]:
# 두 데이터 프레임 병합
import numpy as np
a = pd.DataFrame([{'token':'첨성대', 'count':10 },
                  {'token':'휴가', 'count':10 }])
b = pd.DataFrame([['휴가', 20],
                  ['한옥', 30]], columns=['token','count'])
display(a)
display(b)
ab = pd.merge(a, b, 
              on='token', # 두 프레임을 병합할 기준 열이름
              how='outer') # inner, left, right, outer
ab.fillna(0, inplace=True)
ab['count_x']=ab['count_x'].astype('int')
ab['count_y']=ab['count_y'].astype(np.int16)

,token,count
0,첨성대,10
1,휴가,10


,token,count
0,휴가,20
1,한옥,30


In [30]:
df_mrg = pd.merge(df_gj, df_jj, on='token', how='outer')
df_mrg = df_mrg.fillna(0)
df_mrg = df_mrg.rename(columns={'count_x':'경주빈도', 'count_y':'전주빈도'})

In [31]:
df_mrg['경주빈도'] = df_mrg['경주빈도'].astype(np.int16)
df_mrg['전주빈도'] = df_mrg['전주빈도'].astype(np.int16)
df_mrg['빈도합'] = df_mrg['경주빈도'] + df_mrg['전주빈도']
df_mrg = df_mrg.sort_values(by='빈도합', ascending=False)
df_mrg.head()

,token,경주빈도,전주빈도,빈도합
1283,한옥마을,0,304,304
487,일상,295,0,295
36,고고학,292,0,292
191,맛집,41,163,204
390,알라딘,137,54,191


In [32]:
df_mrg['경주비율'] = round(df_mrg['경주빈도']/df_mrg['빈도합']*100, 1)
df_mrg['전주비율'] = round(df_mrg['전주빈도']/df_mrg['빈도합']*100, 1)
df_mrg.to_csv('data/quiz_naver_nouns_count.csv', index=False, encoding='cp949')

In [33]:
df_mrg.shape

(1312, 6)

# 5. 빈도 시각화(워드클라우드, Text)

In [34]:
df_nouns = pd.read_csv('data/quiz_naver_pos_noun.csv', encoding='cp949')
df_nouns.shape

(17003, 3)

In [35]:
# 경주 명사들만 워드클라우드
# 전주 명사들만 워드클라우드
# 경주 명사들 Text plot
# 전주 명사들만 Text plot

# 6. 워드 임베딩
- Word2Vec

In [36]:
df = pd.read_csv('data/quiz_naver.csv', encoding='cp949')